# Setup do SQL Server — **BibliotecaDb**

Fluxo:

1. Ler cada CSV em `data/` com `pandas.read_csv()`
2. Criar o banco e as tabelas no SQL Server (**IDENTITY** nas PKs)
3. Limpar dados existentes (DELETE na ordem das FKs) e republicar a partir dos CSV
4. Resolver **FKs por nome / ISBN / e-mail** com `SELECT` antes de cada `INSERT`
5. Validar com **`SELECT COUNT(*)`** em cada tabela

**Pré-requisitos:** containers no ar (`docker compose up -d`); ficheiro `.env` na raiz do projeto (modelo em `.env.example`) com senha do SA — **valores com `#` devem ir entre aspas**. No Linux, instale **unixODBC** e um driver SQL Server (**Microsoft ODBC 18** ou **FreeTDS** via `tdsodbc`); a primeira célula de código escolhe automaticamente o driver disponível, salvo `MSSQL_ODBC_DRIVER` no `.env`.


In [8]:
"""Imports ODBC dotenv e diretório da pasta data/."""
import os
from pathlib import Path

import pandas as pd
import pyodbc
from dotenv import load_dotenv

_root = Path.cwd()
if _root.name == "notebook":
    _root = _root.parent
load_dotenv(_root / ".env")

DATA_DIR = _root / "data"


def _format_odbc_driver(name: str) -> str:
    n = name.strip()
    if n.startswith("{") and n.endswith("}"):
        return n
    return "{" + n + "}"


def _resolve_odbc_driver() -> str:
    explicit = os.getenv("MSSQL_ODBC_DRIVER", "").strip()
    if explicit:
        return _format_odbc_driver(explicit)
    available = set(pyodbc.drivers())
    for candidate in (
        "ODBC Driver 18 for SQL Server",
        "ODBC Driver 17 for SQL Server",
        "ODBC Driver 13 for SQL Server",
        "FreeTDS",
    ):
        if candidate in available:
            return _format_odbc_driver(candidate)
    raise RuntimeError(
        "Nenhum driver ODBC para SQL Server encontrado. "
        "Instale: sudo apt install msodbcsql18 (Microsoft) ou sudo apt install tdsodbc (FreeTDS), "
        "ou defina MSSQL_ODBC_DRIVER no .env."
    )


ODBC_DRIVER = _resolve_odbc_driver()
SERVER = os.getenv("MSSQL_JDBC_HOST", "localhost")
PORT = os.getenv("MSSQL_JDBC_PORT", "1433")
USER = os.getenv("MSSQL_JDBC_USER", "sa")
PASSWORD = os.getenv("MSSQL_JDBC_PASSWORD") or os.getenv("MSSQL_SA_PASSWORD")

if not PASSWORD:
    raise RuntimeError("Defina MSSQL_JDBC_PASSWORD ou MSSQL_SA_PASSWORD no arquivo .env")


def string_conexao(database: str) -> str:
    if "FreeTDS" in ODBC_DRIVER:
        return (
            f"DRIVER={ODBC_DRIVER};"
            f"SERVER={SERVER};PORT={PORT};"
            f"DATABASE={database};"
            f"UID={USER};PWD={PASSWORD};"
            "TDS_Version=7.4;"
        )
    return (
        f"DRIVER={ODBC_DRIVER};"
        f"SERVER={SERVER},{PORT};"
        f"DATABASE={database};"
        f"UID={USER};PWD={PASSWORD};"
        "TrustServerCertificate=yes;"
    )


def parse_date_optional(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    if isinstance(val, pd.Timestamp):
        return val.date()
    s = str(val).strip()
    if s == "" or s.lower() == "nan":
        return None
    ts = pd.to_datetime(s, errors="coerce")
    if pd.isna(ts):
        return None
    return ts.date()


print("Driver ODBC:", ODBC_DRIVER)
print("Servidor:", f"{SERVER},{PORT}")
print("Pasta CSV:", DATA_DIR.resolve())


Driver ODBC: {FreeTDS}
Servidor: localhost,1433
Pasta CSV: /mnt/c/source/trabalho2ed-minio-sql/data


In [9]:
"""Lê os CSV (UTF-8 separador vírgula)."""
expected = ("categoria", "autor", "livro", "membro", "emprestimo", "multa")
for nome in expected:
    p = DATA_DIR / f"{nome}.csv"
    if not p.exists():
        raise FileNotFoundError(f"CSV não encontrado: {p}")

df_categoria = pd.read_csv(DATA_DIR / "categoria.csv", encoding="utf-8")
df_autor = pd.read_csv(DATA_DIR / "autor.csv", encoding="utf-8")
df_livro = pd.read_csv(DATA_DIR / "livro.csv", encoding="utf-8")
df_membro = pd.read_csv(DATA_DIR / "membro.csv", encoding="utf-8")
df_emprestimo = pd.read_csv(DATA_DIR / "emprestimo.csv", encoding="utf-8")
df_multa = pd.read_csv(DATA_DIR / "multa.csv", encoding="utf-8")

print("Linhas lidas:")
print(f"  categoria: {len(df_categoria)}")
print(f"  autor: {len(df_autor)}")
print(f"  livro: {len(df_livro)}")
print(f"  membro: {len(df_membro)}")
print(f"  emprestimo: {len(df_emprestimo)}")
print(f"  multa: {len(df_multa)}")


Linhas lidas:
  categoria: 5
  autor: 10
  livro: 20
  membro: 15
  emprestimo: 30
  multa: 8


In [10]:
"""Cria o banco BibliotecaDb (master)."""
cn = pyodbc.connect(string_conexao("master"), autocommit=True)
cur = cn.cursor()
try:
    cur.execute(
        """
        IF NOT EXISTS (SELECT 1 FROM sys.databases WHERE name = N'BibliotecaDb')
        BEGIN
            CREATE DATABASE BibliotecaDb;
        END
        """
    )
    print("Banco BibliotecaDb verificado ou criado.")
finally:
    cur.close()
    cn.close()


Banco BibliotecaDb verificado ou criado.


In [11]:
"""Cria as tabelas com IDENTITY e FKs (idempotente)."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()
try:
    ddl = [
        """
        IF OBJECT_ID(N'dbo.Categoria', N'U') IS NULL
        CREATE TABLE dbo.Categoria (
            id_categoria INT IDENTITY(1,1) NOT NULL PRIMARY KEY,
            nome NVARCHAR(100) NOT NULL,
            descricao NVARCHAR(255) NULL
        );
        """,
        """
        IF OBJECT_ID(N'dbo.Autor', N'U') IS NULL
        CREATE TABLE dbo.Autor (
            id_autor INT IDENTITY(1,1) NOT NULL PRIMARY KEY,
            nome NVARCHAR(150) NOT NULL,
            nacionalidade NVARCHAR(100) NULL,
            data_nascimento DATE NULL
        );
        """,
        """
        IF OBJECT_ID(N'dbo.Livro', N'U') IS NULL
        CREATE TABLE dbo.Livro (
            id_livro INT IDENTITY(1,1) NOT NULL PRIMARY KEY,
            titulo NVARCHAR(200) NOT NULL,
            isbn NVARCHAR(20) NULL UNIQUE,
            ano_publicacao INT NULL,
            id_categoria INT NOT NULL,
            id_autor INT NOT NULL,
            CONSTRAINT FK_Livro_Categoria FOREIGN KEY (id_categoria)
                REFERENCES dbo.Categoria (id_categoria),
            CONSTRAINT FK_Livro_Autor FOREIGN KEY (id_autor)
                REFERENCES dbo.Autor (id_autor)
        );
        """,
        """
        IF OBJECT_ID(N'dbo.Membro', N'U') IS NULL
        CREATE TABLE dbo.Membro (
            id_membro INT IDENTITY(1,1) NOT NULL PRIMARY KEY,
            nome NVARCHAR(150) NOT NULL,
            email NVARCHAR(150) NOT NULL UNIQUE,
            telefone NVARCHAR(20) NULL,
            data_cadastro DATE NOT NULL
        );
        """,
        """
        IF OBJECT_ID(N'dbo.Emprestimo', N'U') IS NULL
        CREATE TABLE dbo.Emprestimo (
            id_emprestimo INT IDENTITY(1,1) NOT NULL PRIMARY KEY,
            id_livro INT NOT NULL,
            id_membro INT NOT NULL,
            data_emprestimo DATE NOT NULL,
            data_devolucao_prevista DATE NOT NULL,
            data_devolucao_real DATE NULL,
            status NVARCHAR(20) NOT NULL,
            CONSTRAINT FK_Emprestimo_Livro FOREIGN KEY (id_livro)
                REFERENCES dbo.Livro (id_livro),
            CONSTRAINT FK_Emprestimo_Membro FOREIGN KEY (id_membro)
                REFERENCES dbo.Membro (id_membro),
            CONSTRAINT CK_Emprestimo_Status CHECK (
                status IN (N'ativo', N'devolvido', N'atrasado')
            )
        );
        """,
        """
        IF OBJECT_ID(N'dbo.Multa', N'U') IS NULL
        CREATE TABLE dbo.Multa (
            id_multa INT IDENTITY(1,1) NOT NULL PRIMARY KEY,
            id_emprestimo INT NOT NULL,
            valor DECIMAL(10,2) NOT NULL,
            pago BIT NOT NULL CONSTRAINT DF_Multa_pago DEFAULT (0),
            data_geracao DATE NOT NULL,
            CONSTRAINT FK_Multa_Emprestimo FOREIGN KEY (id_emprestimo)
                REFERENCES dbo.Emprestimo (id_emprestimo)
        );
        """,
    ]
    for stmt in ddl:
        cur.execute(stmt)
    cn.commit()
    print("Tabelas verificadas ou criadas.")
except Exception:
    cn.rollback()
    raise
finally:
    cur.close()
    cn.close()


Tabelas verificadas ou criadas.


In [12]:
"""Remove linhas antigas e reinicia IDENTITY para nova carga a partir dos CSV."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()
try:
    cur.execute(
        """
        DELETE FROM dbo.Multa;
        DELETE FROM dbo.Emprestimo;
        DELETE FROM dbo.Livro;
        DELETE FROM dbo.Membro;
        DELETE FROM dbo.Autor;
        DELETE FROM dbo.Categoria;
        """
    )
    for tbl in ("Multa", "Emprestimo", "Livro", "Membro", "Autor", "Categoria"):
        cur.execute(f"DBCC CHECKIDENT ('dbo.{tbl}', RESEED, 0)")
    cn.commit()
    print("Tabelas esvaziadas e IDENTITY reiniciado.")
except Exception:
    cn.rollback()
    raise
finally:
    cur.close()
    cn.close()


Tabelas esvaziadas e IDENTITY reiniciado.


In [13]:
"""Insere Categoria e Autor diretamente dos DataFrames."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()
try:
    cur.executemany(
        "INSERT INTO dbo.Categoria (nome, descricao) VALUES (?, ?)",
        df_categoria[["nome", "descricao"]].values.tolist(),
    )
    cur.executemany(
        """
        INSERT INTO dbo.Autor (nome, nacionalidade, data_nascimento)
        VALUES (?, ?, ?)
        """,
        [
            (
                row.nome,
                row.nacionalidade,
                parse_date_optional(row.data_nascimento),
            )
            for row in df_autor.itertuples(index=False)
        ],
    )
    cn.commit()
    print(f"Inseridas {len(df_categoria)} categorias e {len(df_autor)} autores.")
finally:
    cur.close()
    cn.close()


Inseridas 5 categorias e 10 autores.


In [14]:
"""Insere Livro resolvendo id_categoria e id_autor pelo nome."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()


def buscar_id_categoria(nome: str) -> int:
    cur.execute("SELECT id_categoria FROM dbo.Categoria WHERE nome = ?", nome)
    r = cur.fetchone()
    if not r:
        raise KeyError(f"Categoria não encontrada: {nome!r}")
    return int(r[0])


def buscar_id_autor(nome: str) -> int:
    cur.execute("SELECT id_autor FROM dbo.Autor WHERE nome = ?", nome)
    r = cur.fetchone()
    if not r:
        raise KeyError(f"Autor não encontrado: {nome!r}")
    return int(r[0])


try:
    rows_livro = []
    for row in df_livro.itertuples(index=False):
        ic = buscar_id_categoria(row.nome_categoria)
        ia = buscar_id_autor(row.nome_autor)
        rows_livro.append((row.titulo, row.isbn, int(row.ano_publicacao), ic, ia))
    cur.executemany(
        """
        INSERT INTO dbo.Livro (titulo, isbn, ano_publicacao, id_categoria, id_autor)
        VALUES (?, ?, ?, ?, ?)
        """,
        rows_livro,
    )
    cn.commit()
    print(f"Inseridos {len(rows_livro)} livros.")
finally:
    cur.close()
    cn.close()


Inseridos 20 livros.


In [15]:
"""Insere Membro."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()
try:
    cur.executemany(
        """
        INSERT INTO dbo.Membro (nome, email, telefone, data_cadastro)
        VALUES (?, ?, ?, ?)
        """,
        [
            (
                row.nome,
                row.email,
                row.telefone,
                parse_date_optional(row.data_cadastro),
            )
            for row in df_membro.itertuples(index=False)
        ],
    )
    cn.commit()
    print(f"Inseridos {len(df_membro)} membros.")
finally:
    cur.close()
    cn.close()


Inseridos 15 membros.


In [16]:
"""Insere Emprestimo resolvendo id_livro por ISBN e id_membro por e-mail."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()


def buscar_id_livro(isbn: str) -> int:
    cur.execute("SELECT id_livro FROM dbo.Livro WHERE isbn = ?", str(isbn).strip())
    r = cur.fetchone()
    if not r:
        raise KeyError(f"Livro não encontrado para ISBN: {isbn!r}")
    return int(r[0])


def buscar_id_membro(email: str) -> int:
    cur.execute("SELECT id_membro FROM dbo.Membro WHERE email = ?", str(email).strip())
    r = cur.fetchone()
    if not r:
        raise KeyError(f"Membro não encontrado para e-mail: {email!r}")
    return int(r[0])


try:
    rows_emp = []
    for row in df_emprestimo.itertuples(index=False):
        il = buscar_id_livro(row.isbn_livro)
        im = buscar_id_membro(row.email_membro)
        rows_emp.append(
            (
                il,
                im,
                parse_date_optional(row.data_emprestimo),
                parse_date_optional(row.data_devolucao_prevista),
                parse_date_optional(row.data_devolucao_real),
                str(row.status).strip(),
            )
        )
    cur.executemany(
        """
        INSERT INTO dbo.Emprestimo (
            id_livro, id_membro,
            data_emprestimo, data_devolucao_prevista, data_devolucao_real,
            status
        ) VALUES (?, ?, ?, ?, ?, ?)
        """,
        rows_emp,
    )
    cn.commit()
    print(f"Inseridos {len(rows_emp)} empréstimos.")
finally:
    cur.close()
    cn.close()


Inseridos 30 empréstimos.


In [17]:
"""Insere Multa resolvendo id_emprestimo por ISBN + e-mail (empréstimo mais recente)."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cn.autocommit = False
cur = cn.cursor()


def buscar_id_emprestimo(isbn: str, email: str) -> int:
    cur.execute(
        """
        SELECT TOP 1 e.id_emprestimo
        FROM dbo.Emprestimo e
        INNER JOIN dbo.Livro l ON l.id_livro = e.id_livro
        INNER JOIN dbo.Membro m ON m.id_membro = e.id_membro
        WHERE l.isbn = ? AND m.email = ?
        ORDER BY e.data_emprestimo DESC, e.id_emprestimo DESC
        """,
        (str(isbn).strip(), str(email).strip()),
    )
    r = cur.fetchone()
    if not r:
        raise KeyError(f"Empréstimo não encontrado para ISBN {isbn!r} e e-mail {email!r}")
    return int(r[0])


try:
    rows_multa = []
    for row in df_multa.itertuples(index=False):
        ie = buscar_id_emprestimo(row.isbn_livro, row.email_membro)
        pago_bit = bool(int(row.pago))
        rows_multa.append(
            (
                ie,
                float(row.valor),
                pago_bit,
                parse_date_optional(row.data_geracao),
            )
        )
    cur.executemany(
        """
        INSERT INTO dbo.Multa (id_emprestimo, valor, pago, data_geracao)
        VALUES (?, ?, ?, ?)
        """,
        rows_multa,
    )
    cn.commit()
    print(f"Inseridas {len(rows_multa)} multas.")
finally:
    cur.close()
    cn.close()


Inseridas 8 multas.


In [18]:
"""Validação: SELECT COUNT(*) por tabela."""
cn = pyodbc.connect(string_conexao("BibliotecaDb"))
cur = cn.cursor()

esperado = {
    "Categoria": len(df_categoria),
    "Autor": len(df_autor),
    "Livro": len(df_livro),
    "Membro": len(df_membro),
    "Emprestimo": len(df_emprestimo),
    "Multa": len(df_multa),
}

try:
    print("=== Validação COUNT(*) ===")
    for nome, exp in esperado.items():
        cur.execute(f"SELECT COUNT(*) FROM dbo.{nome}")
        total = cur.fetchone()[0]
        ok = "OK" if total == exp else "DIVERGENTE"
        print(f"  {nome}: {total} (esperado {exp}) {ok}")
        assert total == exp, f"{nome}: esperado {exp}, obtido {total}"
    print("\nTodas as contagens conferem com os CSV.")
finally:
    cur.close()
    cn.close()


=== Validação COUNT(*) ===
  Categoria: 5 (esperado 5) OK
  Autor: 10 (esperado 10) OK
  Livro: 20 (esperado 20) OK
  Membro: 15 (esperado 15) OK
  Emprestimo: 30 (esperado 30) OK
  Multa: 8 (esperado 8) OK

Todas as contagens conferem com os CSV.
